# 02 · A/B Test Analysis

Three ways to analyze the same experiment: frequentist z-test, Bayesian Beta-Binomial, and CUPED variance reduction. Same data, same effect, different lenses.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import beta as beta_dist

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.figsize': (9, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

In [ ]:
feats = [f'f{i}' for i in range(12)]
df = pd.read_csv(
    'data/criteo-research-uplift-v2.1.csv',
    usecols=['treatment', 'visit'] + feats
)

# Stratified sample: 25k per group for a realistic experiment size
ctrl_s  = df[df.treatment == 0].sample(25_000, random_state=42)
treat_s = df[df.treatment == 1].sample(25_000, random_state=42)
data = pd.concat([ctrl_s, treat_s]).reset_index(drop=True)
del df

y_c = data.loc[data.treatment == 0, 'visit']
y_t = data.loc[data.treatment == 1, 'visit']
n_c, n_t = len(y_c), len(y_t)
print(f"n: {n_c:,} control / {n_t:,} treatment")
print(f"visit rate: {y_c.mean():.3%} ctrl,  {y_t.mean():.3%} treat")

## Frequentist z-test

Pooled SE under H0 for the test statistic; unpooled SE for the confidence interval.

In [ ]:
p_c, p_t = y_c.mean(), y_t.mean()
delta = p_t - p_c

p_pool   = (y_c.sum() + y_t.sum()) / (n_c + n_t)
se_pool  = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
se_ci    = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)

z      = delta / se_pool
p_val  = 2 * (1 - stats.norm.cdf(abs(z)))
ci_lo, ci_hi = delta - 1.96*se_ci, delta + 1.96*se_ci

print(f"Lift:    {delta:+.3%}")
print(f"95% CI:  [{ci_lo:.3%}, {ci_hi:.3%}]")
print(f"z = {z:.2f},  p = {p_val:.2e}")

## Bayesian Beta-Binomial

Prior: Beta(1, 1), uniform over [0, 1]. Posterior: Beta(1 + successes, 1 + failures). P(treatment > control) estimated via Monte Carlo.

In [ ]:
s_c, s_t = int(y_c.sum()), int(y_t.sum())

mc_c = rng.beta(1 + s_c, 1 + n_c - s_c, 200_000)
mc_t = rng.beta(1 + s_t, 1 + n_t - s_t, 200_000)

prob_better = (mc_t > mc_c).mean()
mc_diff = mc_t - mc_c
ci_bayes = (np.percentile(mc_diff, 2.5), np.percentile(mc_diff, 97.5))

print(f"P(treatment > control) = {prob_better:.1%}")
print(f"95% credible interval: [{ci_bayes[0]:.3%}, {ci_bayes[1]:.3%}]")

# Posterior density plot
post_c = beta_dist(1 + s_c, 1 + n_c - s_c)
post_t = beta_dist(1 + s_t, 1 + n_t - s_t)
x = np.linspace(p_c - 6*se_ci, p_t + 6*se_ci, 1000)

fig, ax = plt.subplots()
ax.fill_between(x, post_c.pdf(x), alpha=0.45, label=f'Control  ({p_c:.3%})')
ax.fill_between(x, post_t.pdf(x), alpha=0.45, label=f'Treatment ({p_t:.3%})')
ax.set_xlabel('visit rate')
ax.set_ylabel('density')
ax.set_title(f'Posterior distributions  |  P(B > A) = {prob_better:.1%}')
ax.legend()
plt.tight_layout()
plt.show()

## CUPED

CUPED (Controlled-experiment Using Pre-Experiment Data) adjusts the outcome by a pre-experiment covariate:

$$Y_{\text{cuped}} = Y - \theta (X - \bar{X}), \quad \theta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$$

This removes variance correlated with X. If treatment is randomized, E[X] is equal across groups and the point estimate is preserved asymptotically. In finite samples, residual covariate imbalance can shift the estimate (CUPED correcting for that imbalance, not a bug).

Variance reduction scales with $\rho^2$. First, find which feature is most correlated with `visit`.

In [ ]:
# Compute correlations in control group only (no treatment contamination)
ctrl_data = data[data.treatment == 0]
corr = ctrl_data[feats].corrwith(ctrl_data['visit']).abs().sort_values(ascending=False)
best = corr.idxmax()
print("Feature correlations with visit (control group):")
print(corr.to_string())
print(f"\nUsing: {best}  (|ρ| = {corr[best]:.4f})")

In [ ]:
X = data[best]
Y = data['visit']
theta = Y.cov(X) / X.var()
data = data.assign(visit_cuped=Y - theta * (X - X.mean()))

y_c_adj = data.loc[data.treatment == 0, 'visit_cuped']
y_t_adj = data.loc[data.treatment == 1, 'visit_cuped']

delta_adj = y_t_adj.mean() - y_c_adj.mean()
se_cuped  = np.sqrt(y_c_adj.var()/n_c + y_t_adj.var()/n_t)
z_cuped   = delta_adj / se_cuped
p_cuped   = 2 * (1 - stats.norm.cdf(abs(z_cuped)))
ci_cuped  = (delta_adj - 1.96*se_cuped, delta_adj + 1.96*se_cuped)
var_red   = 1 - se_cuped**2 / se_ci**2

# Covariate imbalance explains any gap between CUPED and raw estimates
x_imbalance  = data.loc[data.treatment==1, best].mean() - data.loc[data.treatment==0, best].mean()
est_shift    = -theta * x_imbalance

print(f"Lift (adjusted): {delta_adj:+.3%}  (raw: {delta:+.3%},  shift from X imbalance: {est_shift:+.3%})")
print(f"95% CI:          [{ci_cuped[0]:.3%}, {ci_cuped[1]:.3%}]")
print(f"z = {z_cuped:.2f},  p = {p_cuped:.2e}")
print(f"Variance reduction: {var_red:.1%}  (SE: {se_ci:.5f} → {se_cuped:.5f})")

## All three methods compared

In [ ]:
methods = ['Frequentist', 'Bayesian', 'CUPED']
deltas  = [delta,   mc_diff.mean(), delta_adj]
lows    = [ci_lo,   ci_bayes[0],    ci_cuped[0]]
highs   = [ci_hi,   ci_bayes[1],    ci_cuped[1]]

fig, ax = plt.subplots(figsize=(9, 3))
for i, (m, d, lo, hi) in enumerate(zip(methods, deltas, lows, highs)):
    ax.errorbar(d*100, i, xerr=[[(d-lo)*100], [(hi-d)*100]],
                fmt='o', capsize=5, linewidth=2)
    ax.text(hi*100 + 0.02, i, f'[{lo:.3%}, {hi:.3%}]', va='center', fontsize=8)

ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xlabel('Lift in visit rate (pp)')
ax.set_title('95% confidence / credible intervals')
plt.tight_layout()
plt.show()

print(f"\n{'Method':<14} {'Lift':>8} {'CI lo':>8} {'CI hi':>8} {'p-value':>10}")
print(f"{'Frequentist':<14} {delta:>8.3%} {ci_lo:>8.3%} {ci_hi:>8.3%} {p_val:>10.2e}")
print(f"{'Bayesian':<14} {mc_diff.mean():>8.3%} {ci_bayes[0]:>8.3%} {ci_bayes[1]:>8.3%} {'N/A':>10}")
print(f"{'CUPED':<14} {delta_adj:>8.3%} {ci_cuped[0]:>8.3%} {ci_cuped[1]:>8.3%} {p_cuped:>10.2e}")

## Key takeaways

- Frequentist and Bayesian give the same point estimate and nearly identical intervals. The difference is interpretation: a p-value vs. a direct probability that treatment is better.
- CUPED's point estimate may differ from the raw estimate when the covariate is imbalanced across groups (common in small samples). With large, well-balanced samples it converges to the raw estimate while narrowing the CI.
- Variance reduction from CUPED scales with $\rho^2$. Here $\rho \approx 0.45$ gives ~20% reduction in variance, which is meaningful. With $\rho < 0.1$, it's barely worth the complexity.
- Strong CUPED covariates in practice: prior-period outcome (last week's visit rate), segment membership, account age. The feature has to be recorded before randomization.